# 👥 Olist E-Commerce Analytics — Part 3: Customer Segmentation (RFM) & Cohort Retention
**Author:** Data Analytics Portfolio
**Objective:**
1. Calculate Recency, Frequency, and Monetary (RFM) metrics for 93k+ unique customers.
2. Segment customers into actionable behavioral tiers (Champions, Loyal, At Risk, Hibernating, etc.).
3. Build Monthly Cohort Retention Matrices (Month 0 to Month 12).
4. Formulate targeted CRM and marketing retention strategies to increase Customer Lifetime Value (CLV).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

rfm = pd.read_parquet('data/processed/rfm_customer_segments.parquet')
retention_matrix = pd.read_parquet('data/processed/cohort_retention_matrix.parquet')

print(f"Loaded RFM Data for {len(rfm):,} Unique Customers.")


## 1. RFM Segment Distribution & Value Concentration
We profile each behavioral cluster across customer count, average spend, recency, and total GMV contribution.


In [ ]:
seg_profile = rfm.groupby('customer_segment').agg(
    customer_count=('customer_unique_id', 'count'),
    total_spend=('total_spend', 'sum'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('total_spend', 'mean')
).reset_index().sort_values('total_spend', ascending=False)

seg_profile['customer_pct'] = (seg_profile['customer_count'] / seg_profile['customer_count'].sum()) * 100
seg_profile['revenue_pct'] = (seg_profile['total_spend'] / seg_profile['total_spend'].sum()) * 100

seg_profile


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=seg_profile, y='customer_segment', x='customer_count', ax=ax1, palette='Blues_r')
ax1.set_title("Customer Count by Segment", fontsize=13, fontweight='bold')
ax1.set_xlabel("Number of Customers")
ax1.set_ylabel("Segment")

sns.barplot(data=seg_profile, y='customer_segment', x='total_spend', ax=ax2, palette='Greens_r')
ax2.set_title("Total Revenue Contribution by Segment (R$)", fontsize=13, fontweight='bold')
ax2.set_xlabel("Total Spend (R$)")
ax2.set_ylabel("")

plt.tight_layout()
plt.show()


## 2. Monthly Cohort Retention Analysis
The cohort matrix tracks customer purchasing behavior over time, following monthly acquisition cohorts from Month 0 to Month 12.


In [ ]:
cols_to_plot = [c for c in retention_matrix.columns if int(c) <= 12]
retention_subset = retention_matrix[cols_to_plot]

plt.figure(figsize=(16, 10))
sns.heatmap(retention_subset, annot=True, fmt=".1f", cmap="Reds", vmin=0, vmax=2.5, cbar_kws={'label': 'Retention Rate (%)'})
plt.title("Monthly Customer Retention Matrix (%) — Olist Marketplace", fontsize=14, fontweight='bold')
plt.xlabel("Cohort Index (Months Passed Since First Purchase)")
plt.ylabel("Acquisition Cohort Month")
plt.tight_layout()
plt.show()


## 3. Targeted Lifecycle Marketing Playbook
Based on our RFM and Cohort findings, we establish 4 targeted customer marketing playbooks:

| Segment | Share of Base | Strategic Marketing Action | Expected Business Impact |
|---|---|---|---|
| **Champions** | ~1.5% | VIP concierge, exclusive early product drops, referral reward bonuses | Maximize advocacy and word-of-mouth growth |
| **Loyal Customers** | ~2.5% | Cross-category product bundles, loyalty tier milestones | Increase average basket size (+15%) |
| **High-Value New / Recent** | ~12% | Day 7 & Day 21 onboarding drip emails, second-purchase discount vouchers | Convert one-time high spenders into repeat buyers |
| **At Risk / Hibernating** | ~40% | Win-back campaigns featuring dynamic product recommendations and free shipping | Recover 3–5% of lapsed customers |
